<a href="https://colab.research.google.com/github/SilverLi123/RL-Learning/blob/main/notebooks/unit3/unit3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unit 3: Deep Q-Learning with Atari Games 👾 using RL Baselines3 Zoo

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit4/thumbnail.jpg" alt="Unit 3 Thumbnail">

In this notebook, **you'll train a Deep Q-Learning agent** playing Space Invaders using [RL Baselines3 Zoo](https://github.com/DLR-RM/rl-baselines3-zoo), a training framework based on [Stable-Baselines3](https://stable-baselines3.readthedocs.io/en/master/) that provides scripts for training, evaluating agents, tuning hyperparameters, plotting results and recording videos.

We're using the [RL-Baselines-3 Zoo integration, a vanilla version of Deep Q-Learning](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html) with no extensions such as Double-DQN, Dueling-DQN, and Prioritized Experience Replay.

⬇️ Here is an example of what **you will achieve** ⬇️

In [ ]:
%%html
<video controls autoplay><source src="https://huggingface.co/ThomasSimonini/ppo-SpaceInvadersNoFrameskip-v4/resolve/main/replay.mp4" type="video/mp4"></video>

### 🎮 Environments:

- [ALE/SpaceInvaders-v5](https://gymnasium.farama.org/environments/atari/space_invaders/) — current Gymnasium Atari environment ID.

This notebook configures `frameskip: 1` and `repeat_action_probability: 0.0`; Stable-Baselines3’s Atari wrapper then applies four-frame skipping.

## Objectives of this notebook 🏆
At the end of the notebook, you will:
- Be able to understand deeper **how RL Baselines3 Zoo works**.
- Be able to **push your trained agent and the code to the Hub** with a nice video replay and an evaluation score 🔥.




## This notebook is from Deep Reinforcement Learning Course
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Deep RL Course illustration"/>

In this free course, you will:

- 📖 Study Deep Reinforcement Learning in **theory and practice**.
- 🧑‍💻 Learn to **use famous Deep RL libraries** such as Stable Baselines3, RL Baselines3 Zoo, CleanRL and Sample Factory 2.0.
- 🤖 Train **agents in unique environments**

And more check 📚 the syllabus 👉 https://simoninithomas.github.io/deep-rl-course

Don’t forget to **<a href="http://eepurl.com/ic5ZUD">sign up to the course</a>** (we are collecting your email to be able to **send you the links when each Unit is published and give you information about the challenges and updates).**


The best way to keep in touch is to join our discord server to exchange with the community and with us 👉🏻 https://discord.gg/ydHrjt3WP5

## Prerequisites 🏗️
Before diving into the notebook, you need to:

🔲 📚 **[Study Deep Q-Learning by reading Unit 3](https://huggingface.co/deep-rl-course/unit3/introduction)**  🤗

We're constantly trying to improve our tutorials, so **if you find some issues in this notebook**, please [open an issue on the Github Repo](https://github.com/huggingface/deep-rl-class/issues).

# Let’s train a Deep Q-Learning agent playing Atari Space Invaders 👾 and upload it to the Hub.

We recommend using Google Colab for this hands-on so you can focus on learning and experimenting without managing a local RL environment.

The original course certification note used a target score of `>= 200` (`mean_reward - std_reward`) for the legacy `SpaceInvadersNoFrameskip-v4` environment. This notebook now uses `ALE/SpaceInvaders-v5`, so scores are not directly comparable. The course leaderboard is no longer operational; treat the old threshold as historical guidance.

## An advice 💡

Save a copy of this notebook to Google Drive or download it so your edits are preserved if the Colab runtime disconnects.

This notebook is configured for `1e6` training steps. Check the assigned accelerator with `!nvidia-smi`; training time varies with the GPU, runtime load, and library versions. Long runs may exceed the available Colab session.

## Set the GPU 💪
- To **accelerate the agent's training, we'll use a GPU**. To do that, go to `Runtime > Change Runtime type`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step1.jpg" alt="GPU Step 1">

- `Hardware Accelerator > GPU`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step2.jpg" alt="GPU Step 2">

# Install RL-Baselines3 Zoo and its dependencies 📚

The setup below upgrades the RL packages and installs Atari support plus the system tools needed for rendering and video capture. The latest `rl_zoo3` release brings `huggingface_sb3`, which currently requires `huggingface_hub<1.0`; this may conflict with Transformers, Diffusers, or Gradio preinstalled in Colab. This notebook’s RL Zoo workflow uses the Hub 0.x version. Read pip’s conflict report and use the version-check cell below to confirm the RL packages installed.

In [ ]:
%pip install --upgrade "rl_zoo3[plots]" "stable-baselines3[extra]" "gymnasium[atari]" "pyvirtualdisplay" "moviepy>=1.0.0"


In [ ]:
!apt-get update -qq
!apt-get install -y -qq xvfb ffmpeg python3-opengl swig cmake


The current Gymnasium Atari extra installs `ale-py` and Atari ROM support, so the separate `gymnasium[accept-rom-license]` command from older course versions is no longer needed. Run the dependency install above first, then run the version check below. The current RL Zoo requires Python 3.10 or newer and supports Python 3.10–3.13; Colab’s Python version is printed by the check cell.

In [ ]:
# Confirm the Colab Python runtime and the installed RL package versions.
import sys
from importlib.metadata import version

print("Python:", sys.version.split()[0])
for package in ("rl_zoo3", "stable-baselines3", "sb3-contrib", "gymnasium", "ale-py", "huggingface_sb3", "huggingface_hub", "torch", "numpy", "pyvirtualdisplay", "moviepy"):
    print(f"{package}: {version(package)}")


## Create a virtual display 🔽

Video capture in Colab needs a virtual screen to render Atari frames. The setup cells above install `xvfb`, `ffmpeg`, `python3-opengl`, and `pyvirtualdisplay`. Run the next cell before training or recording; it safely reuses the display if run again.

In [ ]:
# Start a virtual display for rendering and video capture in Colab.
from pyvirtualdisplay import Display

if "virtual_display" not in globals() or not virtual_display.is_alive():
    virtual_display = Display(visible=False, size=(1400, 900))
    virtual_display.start()

print("Virtual display is running:", virtual_display.is_alive())

## Train a Deep Q-Learning agent on Space Invaders 👾

Create the Zoo hyperparameter file `dqn.yml` with the code cell below, then run the training command in the following section. The config targets the current Gymnasium ID `ALE/SpaceInvaders-v5` and uses 1 million timesteps.

`frameskip: 1` leaves frame skipping to Stable-Baselines3’s Atari wrapper (four frames); `repeat_action_probability: 0.0` disables sticky actions to match the classic no-sticky-action setup.

In [ ]:
%%writefile dqn.yml
ALE/SpaceInvaders-v5:
  env_kwargs:
    frameskip: 1
    repeat_action_probability: 0.0
  env_wrapper:
    - stable_baselines3.common.atari_wrappers.AtariWrapper
  frame_stack: 4
  policy: "CnnPolicy"
  n_timesteps: !!float 1e6
  buffer_size: 100000
  learning_rate: !!float 1e-4
  batch_size: 32
  learning_starts: 100000
  target_update_interval: 1000
  train_freq: 4
  gradient_steps: 1
  exploration_fraction: 0.1
  exploration_final_eps: 0.01
  optimize_memory_usage: false

The configuration above does the following:

- `AtariWrapper` resizes and grayscales frames and applies four-frame skipping.
- `CnnPolicy` uses a convolutional neural network for image observations.
- `n_timesteps: 1e6` sets the training budget to one million environment steps.
- `buffer_size: 100000` is the replay buffer capacity; the agent samples past transitions from it during learning.

In terms of hyperparameters optimization, my advice is to focus on these 3 hyperparameters:
- `learning_rate`
- `buffer_size (Experience Memory size)`
- `batch_size`

As a good practice, you need to **check the documentation to understand what each hyperparameters does**: https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html#parameters



2. Start training and save checkpoints in the `logs/` folder 📁

Use `--algo dqn` for DQN, `--env ALE/SpaceInvaders-v5` for the environment, `-f logs/` for output, and `--conf-file dqn.yml` to load the configuration.

In [ ]:
!python -m rl_zoo3.train --algo dqn --env ALE/SpaceInvaders-v5 --conf-file dqn.yml -f logs/

#### Solution

In [ ]:
!python -m rl_zoo3.train --algo dqn --env ALE/SpaceInvaders-v5 --conf-file dqn.yml -f logs/

## Let's evaluate our agent 👀
- RL-Baselines3-Zoo provides `enjoy.py`, a python script to evaluate our agent. In most RL libraries, we call the evaluation script `enjoy.py`.
- Let's evaluate it for 5000 timesteps 🔥

In [ ]:
!python -m rl_zoo3.enjoy --algo dqn --env ALE/SpaceInvaders-v5 --no-render --n-timesteps 5000 --folder logs/

#### Solution

In [ ]:
!python -m rl_zoo3.enjoy --algo dqn --env ALE/SpaceInvaders-v5 --no-render --n-timesteps 5000 --folder logs/

## Publish our trained model on the Hub 🚀
Now that we saw we got good results after the training, we can publish our trained model on the hub 🤗 with one line of code.

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit3/space-invaders-model.gif" alt="Space Invaders model">

Using `rl_zoo3.push_to_hub` evaluates the trained agent, records a replay, creates a model card, and uploads the files to your Hugging Face repository.

- Showcase the model and its evaluation result.
- View the replay video.
- Share the agent and its files with others.

The course leaderboard linked in older notebook versions is no longer operational; the model card remains the place to review the evaluation result.

To be able to share your model with the community there are three more steps to follow:

1️⃣ (If it's not already done) create an account to HF ➡ https://huggingface.co/join

2️⃣ Sign in and then, you need to store your authentication token from the Hugging Face website.
- Create a new token (https://huggingface.co/settings/tokens) **with write role**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/create-token.jpg" alt="Create HF Token">

- Copy the token
- Run the cell below and past the token

In [ ]:
from huggingface_hub import notebook_login # To log to our Hugging Face account to be able to upload models to the Hub.
notebook_login()
!git config --global credential.helper store

If you don't want to use a Google Colab or a Jupyter Notebook, you need to use this command instead: `huggingface-cli login`

3️⃣ We're now ready to push our trained agent to the 🤗 Hub 🔥

Let's run push_to_hub.py file to upload our trained agent to the Hub.

`--repo-name `: The name of the repo

`-orga`: Your Hugging Face username

`-f`: Where the trained model folder is (in our case `logs`)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit3/select-id.png" alt="Select Id">

In [ ]:
# Patch the known huggingface_sb3 video-path bug for this Colab runtime.
from pathlib import Path
import huggingface_sb3

package_file = Path(huggingface_sb3.__file__).with_name('push_to_hub.py')
source = package_file.read_text()
old = 'inp = env.video_recorder.path'
new = 'inp = env.video_path'
if old in source:
    package_file.write_text(source.replace(old, new, 1))
    print('Applied video-path compatibility fix:', package_file)
elif new in source:
    print('Video-path compatibility fix is already applied:', package_file)
else:
    raise RuntimeError(f'Expected video-path line was not found in {package_file}')

!python -m rl_zoo3.push_to_hub --algo dqn --env ALE/SpaceInvaders-v5 --repo-name dqn-ALE-SpaceInvaders-v5 -orga Silver321 -f logs/

#### Solution

In [ ]:
!python -m rl_zoo3.push_to_hub --algo dqn --env ALE/SpaceInvaders-v5 --repo-name dqn-ALE-SpaceInvaders-v5 -orga YOUR_HF_USERNAME -f logs/

###.

Congrats 🥳 after you run the upload command, open the repository URL printed by the script (for example, `https://huggingface.co/YOUR_HF_USERNAME/dqn-ALE-SpaceInvaders-v5`). There you can view the replay video, model files, evaluation score, and model card.

## Load a powerful trained model 🔥
- The Stable-Baselines3 team uploaded **more than 150 trained Deep Reinforcement Learning agents on the Hub**.

You can find them here: 👉 https://huggingface.co/sb3

Some examples:
- Asteroids: https://huggingface.co/sb3/dqn-AsteroidsNoFrameskip-v4
- Beam Rider: https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4
- Breakout: https://huggingface.co/sb3/dqn-BreakoutNoFrameskip-v4
- Road Runner: https://huggingface.co/sb3/dqn-RoadRunnerNoFrameskip-v4

Let's load an agent playing Beam Rider: https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4

In [ ]:
%%html
<video controls autoplay><source src="https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4/resolve/main/replay.mp4" type="video/mp4"></video>

### Legacy pretrained-agent example

The demo below uses a previously uploaded `BeamRiderNoFrameskip-v4` checkpoint. It is kept as a historical Hub example and is separate from the current `ALE/*-v5` training workflow above. Its old environment ID may not be registered by current Gymnasium, so skip these two demo commands if the legacy ID is unavailable.

In [ ]:
# Download model and save it into the logs/ folder
!python -m rl_zoo3.load_from_hub --algo dqn --env BeamRiderNoFrameskip-v4 -orga sb3 -f rl_trained/

2. Let's evaluate if for 5000 timesteps

In [ ]:
!python -m rl_zoo3.enjoy --algo dqn --env BeamRiderNoFrameskip-v4 -n 5000  -f rl_trained/ --no-render

Why not trying to train your own **Deep Q-Learning Agent playing BeamRiderNoFrameskip-v4? 🏆.**

If you want to try, check https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4#hyperparameters **in the model card, you have the hyperparameters of the trained agent.**

But finding hyperparameters can be a daunting task. Fortunately, we'll see in the next Unit, how we can **use Optuna for optimizing the Hyperparameters 🔥.**


## Some additional challenges 🏆

The best way to learn is to try things yourself. You can train DQN on other current Gymnasium Atari environments, for example:

- `ALE/BeamRider-v5`
- `ALE/Breakout-v5`
- `ALE/Enduro-v5`
- `ALE/Pong-v5`

The leaderboard link in older versions of this notebook is no longer operational. To study a standalone Atari DQN implementation, see [CleanRL’s DQN Atari code](https://github.com/vwxyzjn/cleanrl/blob/master/cleanrl/dqn_atari.py).

________________________________________________________________________
Congrats on finishing this chapter!

If you’re still feel confused with all these elements...it's totally normal! **This was the same for me and for all people who studied RL.**

Take time to really **grasp the material before continuing and try the additional challenges**. It’s important to master these elements and having a solid foundations.

In the next unit, **we’re going to learn about [Optuna](https://optuna.org/)**. One of the most critical task in Deep Reinforcement Learning is to find a good set of training hyperparameters. And Optuna is a library that helps you to automate the search.






### This is a course built with you 👷🏿‍♀️

Finally, we want to improve and update the course iteratively with your feedback. If you have some, please fill this form 👉 https://forms.gle/3HgA7bEHwAmmLfwh9

We're constantly trying to improve our tutorials, so **if you find some issues in this notebook**, please [open an issue on the Github Repo](https://github.com/huggingface/deep-rl-class/issues).

See you on Bonus unit 2! 🔥

### Keep Learning, Stay Awesome 🤗